# Imports

In [1]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
from rasterio.transform import array_bounds
from rasterio.windows import from_bounds
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.ticker as mtick
import matplotlib.colors as mcolors
import seaborn as sns
from rasterio.warp import calculate_default_transform
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.model_selection import GroupKFold, validation_curve, KFold
from sklearn.feature_selection import RFECV, RFE
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.inspection import partial_dependence
import joblib
from pathlib import Path
import os
import warnings, random
warnings.filterwarnings("ignore", category=FutureWarning)

# Kitchen Sink Model

In [2]:
# Full Kitchen Sink Model for Wildfire Burn Severity (dNBR)
# Wildfire RF pipeline

# -----------------------------
# Inputs
# -----------------------------
"""
For each fire, we have:
- a burn perimeter clipped inward by 100m (to remove edge effects from firefighting) 
- an ENVI-formatted file with all data layers stacked and resampled (bilinear) to EMIT spatial resolution
- a mask omitting water and urban classes (from NLCD)
- a raster with NLCD classes nearest-neighbor resampled to EMIT resolution
"""

wildfire_rasters = {
    "Eaton": {
        "burn_perimeter": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/data/allfires_model/burnperimeters/Eaton_WFIGS_Interagency_Perimeters_YearToDate_-8396875942426194654/Perimeters_inward100m.shp",
        "stacked": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Eaton/Eaton_LayerStack_updated20250902",
        "mask": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Eaton/NLCD_Eaton_mask_resampled.tif",
        "nlcd_raster": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Eaton/NLCD_Eaton_resampled.tif"
    },
    "Hughes": {
        "burn_perimeter": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/data/allfires_model/burnperimeters/Hughes_WFIGS_Interagency_Perimeters_YearToDate_7148479240675437113/Perimeters_inward100m.shp",
        "stacked": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Hughes/Hughes_layerstack_updated_20250902",
        "mask": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Hughes/NLCD_Hughes_mask_resampled.tif",
        "nlcd_raster": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Hughes/NLCD_Hughes_resampled.tif"
    },
    "Palisades": {
        "burn_perimeter": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/data/allfires_model/burnperimeters/Palisades_WFIGS_Interagency_Perimeters_YearToDate_2024716492027912883/Perimeters_inward100m.shp",
        "stacked": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Palisades/Palisades_LayerStack_updated20250902",
        "mask": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Palisades/NLCD_Palisades_mask_resampled.tif",
        "nlcd_raster": "/Volumes/GEOG/EMIT_ECOSTRESS_wildfires/updated_allfiresmodel/Palisades/NLCD_Palisades_resampled.tif"
    }
}

# -------------------------------------------
# Define per-fire band mappings
# -------------------------------------------
"""
We include only predictors shared across fires.
"""

base_band_mapping = {
        1: "CWC_20240625",
        #2: "Normalized_CWC_NDVI_20240625",
        #3: "Normalized_CWC_Vfrac_20240625",
        4: "Vfrac_20240625",
        #5: "NDVI_20240625",
        6: "ET_20240625_1038am",
        7: "ESI_20240625_1038am",
        8: "WUE_20240625_1038am",
        9: "ET_20241201_1049am",
        10: "ESI_20241201_1049am",
        11: "WUE_20241201_1049am",
        12: "ET_20231202_1121am",
        13: "ESI_20231202_1121am",
        14: "WUE_20231202_1121am",
        15: "ET_20230409_1028am",
        16: "ESI_20230409_1028am",
        17: "WUE_20230409_1028am",
        18: "ETdiff_Dec24-June24",
        #19: "ETpctchangeJunDec",
        20: "ETdiffDec24-Dec23",
        #21: "ETpctchngDecDec",
        #22: "ETdiffAprDec",
        #23: "ETpctchgAprDec",
        24: "ESIdiff_Dec24-June24",
        #25: "ESIpctchangeJunDec",
        26: "ESIdiffDec24-Dec23",
        #27: "ESIpctchngDecDec",
        #28: "ESIdiffAprDec",
        #29: "ESIpctchgAprDec",
        30: "WUEdiff_Dec24-June24",
        #31: "WUEpctchangeJunDec",
        32: "WUEdiffDec24-Dec23",
        #33: "WUEpctchngDecDec",
        #34: "WUEdiffAprDec",
        #35: "WUEpctchgAprDec",
        36: "AreaSolarRadiation_all",
        37: "AreaSolarRadiation_directonly",
        38: "Elevation",
        39: "Slope",
        #40: "Aspect",
        41: "cosAspect",
        42: "WindSpeed_avg_20250108",
        43: "UGRD_avg_20250108",
        44: "VGRD_avg_20250108",
        45: "WindGust_max_20250108",
        46: "VPD_max_20250107",
        #47: "NLCD", #biinlinear resampling makes this layer incorrect
        #48: "Landfire_LC23_F13_240", #biinlinear resampling makes this layer incorrect
        #49: "preNBR_L8_20250106",
        #50: "postNBR_L8_20250223",
        51: "dNBR",
        52: "ESIdiff_Dec24-Apr23",
        53: "WUEdiff_Dec24-Apr23"
}

band_mappings = {
    "Eaton": base_band_mapping,
    "Hughes": base_band_mapping,
    "Palisades": base_band_mapping,
}

# Get predictor names 
# All fires share the same band mapping, so we can safely use Eaton
predictor_names = sorted([
    name for name in band_mappings["Eaton"].values()
    if name != "dNBR"
])

continuous_vars = predictor_names[:]

# -----------------------------
# NLCD
# -----------------------------
# NLCD code-to-class lookup
nlcd_classes = {
    11: 'Open Water', 12: 'Perennial Ice/Snow', 21: 'Developed, Open Space',
    22: 'Developed, Low Intensity', 23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity', 31: 'Barren Land',
    41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
    52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous', 81: 'Pasture/Hay',
    82: 'Cultivated Crops', 90: 'Woody Wetlands', 95: 'Emergent Herbaceous Wetlands'
}

# -----------------------------
# Load raster and extract valid pixels
# -----------------------------
"""
Load predictor and target bands from a stacked raster.

Returns
-------
predictors : np.ndarray
    3D array (bands, rows, cols) of predictors.
target : np.ndarray
    2D array (rows, cols) of dNBR values.
predictor_names_out : list
    Names of predictors extracted.
profile : dict
    Raster profile metadata.
mask_data : np.ndarray
    Binary mask of valid pixels.
"""
def load_predictor_and_target_bands(raster_path, burn_perimeter, band_map, mask_path, nlcd_path):
    with rasterio.open(raster_path) as src:
        geom = [burn_perimeter.geometry.union_all().__geo_interface__]
        all_bands_data, transform = mask(src, geom, crop=True, nodata=-9999)
        all_bands_data = all_bands_data.astype(float)
        all_bands_data[all_bands_data == -9999] = np.nan
        profile = src.profile
        profile.update({
            "height": all_bands_data.shape[1],
            "width": all_bands_data.shape[2],
            "transform": transform
        })

    with rasterio.open(mask_path) as mask_src:
        mask_clipped, _ = mask(mask_src, geom, crop=True, nodata=0)
        mask_data = mask_clipped[0, :, :]

    with rasterio.open(nlcd_path) as nlcd_src:
        nlcd_clipped, _ = mask(nlcd_src, geom, crop=True, nodata=0)
        nlcd_data = nlcd_clipped[0, :, :]

    predictors, predictor_names_out = [], []
    for band_num, name in band_map.items():
        band_data = all_bands_data[band_num - 1, :, :]
        if name == "dNBR":
            target = band_data
        else:
            predictors.append(band_data)
            predictor_names_out.append(name)

    predictors = np.array(predictors)
    return predictors, target, predictor_names_out, profile, mask_data, nlcd_data

# -----------------------------
# Subsampling mask (~300 m)
# -----------------------------
"""
Generate a boolean mask that selects one random valid pixel per block_size x block_size area.
Helps reduce spatial autocorrelation (≈300 m subsampling).
"""
def random_subsample_mask(mask_data, block_size=5, seed=42):
    rng = np.random.default_rng(seed)  # create seeded RNG
    rows, cols = mask_data.shape
    subsample_mask = np.zeros_like(mask_data, dtype=bool)

    for i in range(0, rows, block_size):
        for j in range(0, cols, block_size):
            block = mask_data[i:i+block_size, j:j+block_size]
            valid_indices = np.argwhere(block == 1)
            if valid_indices.size > 0:
                idx = rng.choice(valid_indices)
                subsample_mask[i + idx[0], j + idx[1]] = True

    return subsample_mask


# -----------------------------
# Loop over fires and extract data
# -----------------------------
"""
For each fire:
1. Read the burn perimeter and stacked raster bands.
2. Extract predictors (all continuous variables) and the target (dNBR) within the burn perimeter.
3. Apply a ~300 m block subsampling mask to reduce spatial autocorrelation,
   keeping one random valid pixel per block -> stored in df_all_unscaled.
4. Also extract all valid pixels (full-scene, no subsampling) -> stored in df_all_full.
5. Record raster metadata (target + profile) in profiles for later georeferencing.
"""

df_all_unscaled, df_all_full, profiles = [], [], {}

for fire, paths in wildfire_rasters.items():
    print(f"Processing {fire}")
    burn_perimeter = gpd.read_file(paths["burn_perimeter"])
    band_map = band_mappings[fire]

    predictors, target, pred_names_fire, profile, mask_data, nlcd_data = load_predictor_and_target_bands(
        paths["stacked"], burn_perimeter, band_map, paths["mask"], paths["nlcd_raster"]
    )

    rows, cols = target.shape
    row_indices, col_indices = np.indices((rows, cols))
    transform = profile["transform"]

    # --- Subsample (~300 m) ---
    subsample_mask = random_subsample_mask((~np.isnan(target)) & (mask_data == 1), block_size=5, seed=42)
    valid_mask = subsample_mask
    flat_predictors = predictors[:, valid_mask].T

    data = {
        "dNBR": target[valid_mask].flatten(),
        "NLCD": nlcd_data[valid_mask].flatten(),
        "NLCD_Class": [nlcd_classes.get(code, "Unknown") for code in nlcd_data[valid_mask].flatten()],
        "Row": row_indices[valid_mask].flatten(),
        "Col": col_indices[valid_mask].flatten(),
        "X": np.array(rasterio.transform.xy(transform, row_indices[valid_mask], col_indices[valid_mask])[0]),
        "Y": np.array(rasterio.transform.xy(transform, row_indices[valid_mask], col_indices[valid_mask])[1]),
        "Fire": [fire] * flat_predictors.shape[0]
    }
    for i, name in enumerate(pred_names_fire):
        data[name] = flat_predictors[:, i]

    df_fire = pd.DataFrame(data).dropna().reset_index(drop=True)
    df_all_unscaled.append(df_fire)

    # --- Full-scene (no subsampling) ---
    full_valid_mask = (~np.isnan(target)) & (mask_data == 1)
    flat_predictors_full = predictors[:, full_valid_mask].T

    data_full = {
        "dNBR": target[full_valid_mask].flatten(),
        "NLCD": nlcd_data[full_valid_mask].flatten(),
        "NLCD_Class": [nlcd_classes.get(code, "Unknown") for code in nlcd_data[full_valid_mask].flatten()],
        "Row": row_indices[full_valid_mask].flatten(),
        "Col": col_indices[full_valid_mask].flatten(),
        "X": np.array(rasterio.transform.xy(transform, row_indices[full_valid_mask], col_indices[full_valid_mask])[0]),
        "Y": np.array(rasterio.transform.xy(transform, row_indices[full_valid_mask], col_indices[full_valid_mask])[1]),
        "Fire": [fire] * flat_predictors_full.shape[0]
    }
    for i, name in enumerate(pred_names_fire):
        data_full[name] = flat_predictors_full[:, i]

    df_all_full.append(pd.DataFrame(data_full).dropna().reset_index(drop=True))
    profiles[fire] = (target, profile)

# -----------------------------
# Scaling
# -----------------------------
"""
We fit the StandardScaler on all fires combined to ensure variables are standardized consistently across fires.
"""
scaler = StandardScaler()
df_unscaled = pd.concat(df_all_unscaled, ignore_index=True)
scaler.fit(df_unscaled[continuous_vars])

df_all_scaled = []
for df_fire in df_all_unscaled:
    df_scaled = df_fire.copy()
    df_scaled[continuous_vars] = scaler.transform(df_scaled[continuous_vars])
    df_all_scaled.append(df_scaled)

df = pd.concat(df_all_scaled, ignore_index=True)
df_full = pd.concat(df_all_full, ignore_index=True)
df_full_scaled = df_full.copy()
df_full_scaled[continuous_vars] = scaler.transform(df_full[continuous_vars])

# -----------------------------
# Modeling
# -----------------------------
"""
Prepare predictors (X) and target (y), then split into training/testing sets.
- Stratify by "Fire" to keep a balanced representation of each fire in train/test.
- Use RandomizedSearchCV to tune Random Forest hyperparameters at first:
    * n_estimators: number of trees
    * max_depth: tree depth
    * min_samples_split / min_samples_leaf: control tree complexity
    * max_features: number of features to consider at each split
- Search over 25 random parameter combinations with 5-fold CV.
- Optimize for R² score and select the best-performing model.
"""
X = df[continuous_vars]
y = df["dNBR"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=df["Fire"]
)

param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [10, 20, 30],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2']
}

rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    scoring='r2'
)
rf.fit(X_train, y_train)
model = rf.best_estimator_

# -----------------------------
# Apply the trained model to the full dataset (all valid pixels)
# -----------------------------
"""
- X_full: all continuous predictors, scaled consistently across fires
- Predicted_dNBR: model output for each pixel
- Residual: difference between prediction and observed dNBR
This gives wall-to-wall predictions (not just test points) so results can be mapped
and residuals analyzed spatially.
"""
X_full = df_full_scaled[continuous_vars]
df_full_scaled["Predicted_dNBR"] = model.predict(X_full)
df_full_scaled["Residual"] = df_full_scaled["Predicted_dNBR"] - df_full_scaled["dNBR"]

print(df_full_scaled[["Fire", "Row", "Col", "X", "Y", "Predicted_dNBR", "dNBR", "Residual"]].sample(5))

Processing Eaton
Processing Hughes
Processing Palisades
            Fire  Row  Col           X          Y  Predicted_dNBR      dNBR  \
43433  Palisades  132  324 -118.508177  34.056920        0.528355  0.484843   
31986  Palisades   81  288 -118.527697  34.084574        0.707756  0.761478   
4168       Eaton   44   88 -118.110177  34.211992        0.522767  0.470629   
37827  Palisades  112   41 -118.661628  34.067765        0.459257  0.340758   
38677  Palisades  115   35 -118.664881  34.066138        0.579420  0.660582   

       Residual  
43433  0.043513  
31986 -0.053722  
4168   0.052138  
37827  0.118499  
38677 -0.081162  


In [3]:
# ----------------------------
# Kitchen Sink Model Performance Metrics
# ----------------------------
"""
Evaluate how well the tuned Random Forest performs:
- Compare training vs. testing R² to check for overfitting
- Report standard error metrics (RMSE, MAE) on the test set
- Show the difference between Train and Test R² as an overfit indicator
- Print the best hyperparameters found via RandomizedSearchCV
- Display the mean CV R² score from the search
"""

# Predict on training and testing data using the best model
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Compute metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

# Print metrics
print("\nKitchen Sink Model (Train vs Test):")
print(f"Train R²:       {r2_train:.3f}")
print(f"Test R²:        {r2_test:.3f}")
print(f"Test RMSE:      {rmse_test:.3f}")
print(f"Test MAE:       {mae_test:.3f}")
print(f"Test - Train R²:{r2_test - r2_train:.3f}")

# Print best parameters and score from RandomizedSearchCV
print("Best RandomizedSearchCV Parameters:")
print(rf.best_params_)

print(f"\nBest R² Score from CV: {rf.best_score_:.3f}")

# ----------------------------
# Track Fire Source in Metrics
# ----------------------------

# To better interpret generalization across fires, evaluate R² per fire in the test set
df_test = df.iloc[X_test.index].copy()
df_test["y_true"] = y_test
df_test["y_pred"] = y_pred_test
per_fire_r2 = df_test.groupby("Fire").apply(lambda g: r2_score(g["y_true"], g["y_pred"]))
print("\nTest R² per fire:\n", per_fire_r2)

# -------------------------------------------
# Print training and testing sample counts
# -------------------------------------------

# Add back fire labels to splits for per-fire counts
train_idx = X_train.index
test_idx = X_test.index

df["Set"] = "Unused"
df.loc[train_idx, "Set"] = "Train"
df.loc[test_idx, "Set"] = "Test"

# Count per fire and set
fire_counts = df.groupby(["Fire", "Set"]).size().unstack(fill_value=0)
print("\nPixel counts per fire and split:")
print(fire_counts)

# Overall count
total_train = len(X_train)
total_test = len(X_test)
print(f"\nTotal training pixels: {total_train}")
print(f"Total testing pixels: {total_test}")
print(f"Total pixels: {total_train + total_test}")


Kitchen Sink Model (Train vs Test):
Train R²:       0.819
Test R²:        0.618
Test RMSE:      0.117
Test MAE:       0.086
Test - Train R²:-0.201
Best RandomizedSearchCV Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 30}

Best R² Score from CV: 0.587

Test R² per fire:
 Fire
Eaton        0.316324
Hughes       0.696957
Palisades    0.637678
dtype: float64

Pixel counts per fire and split:
Set        Test  Train
Fire                  
Eaton       184    429
Hughes      166    387
Palisades   337    787

Total training pixels: 1603
Total testing pixels: 687
Total pixels: 2290


/var/folders/br/pyt6szbd2wb4q6n5wqkffrbc0000gn/T/ipykernel_20606/2051953484.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_fire_r2 = df_test.groupby("Fire").apply(lambda g: r2_score(g["y_true"], g["y_pred"]))


# Save kitchen sink model

In [4]:
# ----------------------------
# Save the Trained Model 
# ----------------------------
"""
- Save the trained Random Forest model (pkl via joblib) so it can be reloaded later
- Export the full prediction DataFrame (with predictors, observed dNBR, predictions, residuals)
  as a Parquet file for efficient storage and downstream analysis/visualization
"""
joblib.dump(model, "/Users/megan/Desktop/runmodel/kitchen_sink_model_2025103.pkl")

# Export
df_full_scaled.to_parquet("/Users/megan/Desktop/runmodel/kitchen_sink_model_20251023.parquet")

df.to_parquet(
    "/Users/megan/Desktop/runmodel/"
    "df_train_scaled_300m.parquet",
    index=False
)
joblib.dump(
    scaler,
    "/Users/megan/Desktop/runmodel/"
    "scaler_allfires.joblib"
)

['/Users/megan/Desktop/runmodel/scaler_allfires.joblib']

# Run and save reduced feature model 

In [5]:
# Train, evaluate, and apply the reduced-feature Random Forest model for dNBR prediction, saving both the fitted model and full-scene prediction outputs.

# ------------------------------------------------------------
# 0) Paths 
# ------------------------------------------------------------
TRAIN_PARQUET = (
    "/Users/megan/Desktop/runmodel/"
    "df_train_scaled_300m.parquet"
)

FULL_PARQUET = (
    "/Users/megan/Desktop/runmodel/"
    "kitchen_sink_model_20251023.parquet"
)

OUT_MODEL_PKL = (
    "/Users/megan/Desktop/runmodel/"
    "reduced_rf_tuned_20251023.pkl"
)

OUT_PRED_PARQUET = (
    "/Users/megan/Desktop/runmodel/"
    "reduced_rf_predictions_20251023.parquet"
)


# ------------------------------------------------------------
# 1) Final reduced predictors 
# ------------------------------------------------------------
reduced_vars_final = [
    "CWC_20240625",
    "ESI_20241201_1049am",
    "ESI_20240625_1038am",
    "WUE_20230409_1028am",
    "WindGust_max_20250108",
    "Elevation",
    "cosAspect",
]


# ------------------------------------------------------------
# 2) Load data
# ------------------------------------------------------------
if not os.path.exists(TRAIN_PARQUET):
    raise FileNotFoundError(f"\nMissing TRAIN_PARQUET:\n  {TRAIN_PARQUET}\n\n")

if not os.path.exists(FULL_PARQUET):
    raise FileNotFoundError(f"\nMissing FULL_PARQUET:\n  {FULL_PARQUET}\n")

df = pd.read_parquet(TRAIN_PARQUET)         # subsampled, scaled training points
df_full_scaled = pd.read_parquet(FULL_PARQUET)  # full-scene, scaled pixels

# Quick sanity checks
required_cols_train = set(["dNBR", "Fire"] + reduced_vars_final)
required_cols_full  = set(["dNBR", "Fire", "Row", "Col", "X", "Y"] + reduced_vars_final)

missing_train = sorted(list(required_cols_train - set(df.columns)))
missing_full  = sorted(list(required_cols_full  - set(df_full_scaled.columns)))

if missing_train:
    raise KeyError(f"Missing columns in TRAIN df: {missing_train}")
if missing_full:
    raise KeyError(f"Missing columns in FULL df_full_scaled: {missing_full}")

# Drop any remaining NaNs in the reduced set 
df = df.dropna(subset=reduced_vars_final + ["dNBR", "Fire"]).reset_index(drop=True)
df_full_scaled = df_full_scaled.dropna(subset=reduced_vars_final + ["dNBR", "Fire"]).reset_index(drop=True)


# ------------------------------------------------------------
# 3) Train/test split (stratified by Fire)
# ------------------------------------------------------------
X = df[reduced_vars_final]
y = df["dNBR"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=df["Fire"]
)


# ------------------------------------------------------------
# 4) Train final reduced tuned RF 
# ------------------------------------------------------------
custom_params_red = {
    "n_estimators": 400,
    "max_depth": 10,
    "min_samples_split": 20,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "oob_score": True,
    "random_state": 42,
    "n_jobs": -1,
}

rf_reduced_tuned = RandomForestRegressor(**custom_params_red)
rf_reduced_tuned.fit(X_train, y_train)


# ------------------------------------------------------------
# 5) Evaluate
# ------------------------------------------------------------
ytr = rf_reduced_tuned.predict(X_train)
yte = rf_reduced_tuned.predict(X_test)

oob = getattr(rf_reduced_tuned, "oob_score_", np.nan)

print("\nReduced Model Performance (Final Tuned):")
print("-------------------------------------------------------")
print(f"OOB R²:             {oob:.3f}")
print(f"Train R²:           {r2_score(y_train, ytr):.3f}")
print(f"Test R²:            {r2_score(y_test, yte):.3f}")
print(f"Test RMSE:          {np.sqrt(mean_squared_error(y_test, yte)):.3f}")
print(f"Test MAE:           {mean_absolute_error(y_test, yte):.3f}")
print(f"Generalization Gap: {r2_score(y_test, yte) - r2_score(y_train, ytr):.3f}")


# ------------------------------------------------------------
# 6) Wall-to-wall prediction on full-scene parquet
# ------------------------------------------------------------
df_full_scaled["Predicted_dNBR_reduced"] = rf_reduced_tuned.predict(df_full_scaled[reduced_vars_final])
df_full_scaled["Residual_reduced"] = df_full_scaled["Predicted_dNBR_reduced"] - df_full_scaled["dNBR"]

print("\nSample predictions:")
print(df_full_scaled[["Fire","Row","Col","X","Y","Predicted_dNBR_reduced","dNBR","Residual_reduced"]]
      .sample(5, random_state=42))


# ------------------------------------------------------------
# 7) Save model + output parquet
# ------------------------------------------------------------
joblib.dump(rf_reduced_tuned, OUT_MODEL_PKL)
df_full_scaled.to_parquet(OUT_PRED_PARQUET, index=False)

print("\nSaved:")
print(f"  Model:   {OUT_MODEL_PKL}")
print(f"  Parquet: {OUT_PRED_PARQUET}")


Reduced Model Performance (Final Tuned):
-------------------------------------------------------
OOB R²:             0.572
Train R²:           0.719
Test R²:            0.599
Test RMSE:          0.120
Test MAE:           0.089
Generalization Gap: -0.120

Sample predictions:
            Fire  Row  Col           X          Y  Predicted_dNBR_reduced  \
34405  Palisades   97   97 -118.631263  34.075898                0.720306   
45038  Palisades  140  111 -118.623672  34.052582                0.658629   
28600  Palisades   49  258 -118.543964  34.101925                0.584057   
33569  Palisades   92  176 -118.588427  34.078609                0.675741   
29860  Palisades   63  246 -118.550471  34.094334                0.416315   

           dNBR  Residual_reduced  
34405  0.952923         -0.232617  
45038  0.715195         -0.056566  
28600  0.610245         -0.026188  
33569  0.884120         -0.208379  
29860  0.467869         -0.051554  

Saved:
  Model:   /Users/megan/Desktop/runmo